In [ ]:
import requests
import json
from bs4 import BeautifulSoup


class Website:
    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        
        # Strip out code and styling tags
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)


target_url = "https://edwarddonner.com"
webpage = Website(target_url)


system_prompt = "You are an assistant that provides a short summary of a website in markdown."
user_prompt = f"You are looking at a website titled {webpage.title}.\n\nContents:\n{webpage.text}"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]


def summarize_via_http(messages_payload):
    # Ollama's local server endpoint
    url = "http://localhost:11434/api/chat"
    
    # The JSON configuration payload sent to Ollama
    payload = {
        "model": "llama3.2",
        "messages": messages_payload,
        "stream": False  
    }
    
    
    response = requests.post(url, json=payload)
    
    
    data = response.json()
    return data['message']['content']


summary = summarize_via_http(messages)
print(summary)